# BUSI: Breast Ultrasound Image Quality Assessment (IQA) & Adaptive Routing
### Modality: Breast Ultrasound (B-mode Sonography with Paired Segmentation Masks)
### Dataset: Breast Ultrasound Images Dataset (BUSI)

---

## 📌 Clinical & Sonographic Objectives
1. **Dataset Characterization:** Map ultrasound cases across `normal` (133), `benign` (437), and `malignant` (210) categories with paired pixel-level tumor segmentation masks.
2. **Identification of Inherent Challenges:** Multiplicative acoustic speckle noise, posterior acoustic shadows, hypoechoic tissue contrast boundaries, and transducer angle/pressure variability.
3. **Multi-Metric IQA Pipeline:**
   - **Sharpness:** Laplacian Variance ($\sigma^2_{\Delta}$), Tenengrad Gradient
   - **Acoustic Speckle:** Speckle Index ($C_s = \sigma / \mu$), Equivalent Number of Looks (ENL = $\mu^2 / \sigma^2$)
   - **Lesion Contrast:** Contrast-to-Noise Ratio (CNR) between lesion mask and background parenchyma
   - **Dynamic Contrast & Dynamic Range:** RMS Contrast ($C_{\text{RMS}}$), Michelson Contrast
   - **Information Content:** Shannon Information Entropy ($H$)
   - **Noise Estimation:** Blind SNR via Median Absolute Deviation (MAD)
4. **Statistical Analysis:** Generate descriptive summaries across sonographic categories, correlation heatmaps, and side-by-side visual montages.
5. **Learned Decision Tree:** Train an explainable adaptive routing model that evaluates image fidelity + CNN prediction confidence before triggering automated diagnostic sign-off or radiologist review.

---
## 🚀 Running on Kaggle
- Click **"Add Input"** on the right sidebar and search for `aryashah2k/breast-ultrasound-images-dataset`.
- The notebook auto-detects `/kaggle/input/` paths and writes outputs to `/kaggle/working/`.


In [ ]:
# Cell 1: Environment Setup & Library Imports
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from scipy.stats import entropy
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
print("Environment and libraries loaded successfully.")


In [ ]:
# Cell 2: Dataset Discovery & Path Configuration
KAGGLE_CANDIDATE_PATHS = [
    "/kaggle/input/breast-ultrasound-images-dataset/Dataset_BUSI_with_masks",
    "/kaggle/input/breast-ultrasound-images-dataset",
    "/kaggle/input/busi-dataset",
    "./sample_data",
    "../sample_data",
]

def locate_busi_dataset():
    for p in KAGGLE_CANDIDATE_PATHS:
        if os.path.exists(p) and len(os.listdir(p)) > 0:
            print(f"[SUCCESS] BUSI located at: {p}")
            return p
    print("[NOTICE] No external Kaggle dataset mounted. Initializing sample ultrasound cohort...")
    return None

DATASET_ROOT = locate_busi_dataset()
OUTPUT_DIR = "/kaggle/working/output" if os.path.exists("/kaggle") else "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Cell 3: Synthetic Ultrasound Generator (Zero-Error Fallback)
def generate_synthetic_ultrasound(target_dir, n_samples=60):
    os.makedirs(target_dir, exist_ok=True)
    classes = ["normal", "benign", "malignant"]
    np.random.seed(42)

    for i in range(n_samples):
        cls = classes[i % len(classes)]
        cls_dir = os.path.join(target_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)

        h, w = 256, 256
        base = np.zeros((h, w), dtype=np.float32)
        base[0:int(h * 0.25), :] = 80.0
        base[int(h * 0.25):int(h * 0.75), :] = 120.0
        base[int(h * 0.75):, :] = 60.0

        mask = np.zeros((h, w), dtype=np.uint8)

        if cls != "normal":
            cx = np.random.randint(int(w * 0.35), int(w * 0.65))
            cy = np.random.randint(int(h * 0.35), int(h * 0.65))

            if cls == "benign":
                rx = np.random.randint(25, 40)
                ry = np.random.randint(15, 25)
                cv2.ellipse(mask, (cx, cy), (rx, ry), 0, 0, 360, 255, -1)
                base[mask > 0] = np.random.uniform(40, 55)
            else:
                rx = np.random.randint(20, 35)
                ry = np.random.randint(25, 45)
                cv2.ellipse(mask, (cx, cy), (rx, ry), 15, 0, 360, 255, -1)
                for _ in range(10):
                    px = cx + np.random.randint(-rx, rx)
                    py = cy + np.random.randint(-ry, ry)
                    cv2.circle(mask, (px, py), np.random.randint(6, 12), 255, -1)
                base[mask > 0] = np.random.uniform(25, 45)
                base[cy + ry:h, max(0, cx - rx):min(w, cx + rx)] *= 0.45

        speckle = np.random.gamma(shape=4.0, scale=0.25, size=(h, w))
        us_img = np.clip(base * speckle, 0, 255).astype(np.uint8)

        if i % 4 == 0:
            us_img = cv2.GaussianBlur(us_img, (5, 5), 0)

        img_name = f"{cls} ({i + 1}).png"
        mask_name = f"{cls} ({i + 1})_mask.png"
        cv2.imwrite(os.path.join(cls_dir, img_name), us_img)
        cv2.imwrite(os.path.join(cls_dir, mask_name), mask)

    return target_dir

if DATASET_ROOT is None:
    DATASET_ROOT = generate_synthetic_ultrasound(os.path.join(OUTPUT_DIR, "synthetic_busi"))


## 1. Ultrasound Characterization & Metadata Parsing
We parse primary ultrasound scans, map paired ground truth lesion segmentation masks, and record classification targets (`normal`, `benign`, `malignant`).


In [ ]:
# Cell 4: Parse BUSI Scans and Pair Ground Truth Masks
image_paths = []
for ext in ("*.png", "*.jpg", "*.jpeg"):
    image_paths.extend(glob.glob(os.path.join(DATASET_ROOT, "**", ext), recursive=True))

primary_scans = [p for p in image_paths if "_mask" not in Path(p).name.lower()]

meta_list = []
for path in primary_scans:
    p = Path(path)
    fname = p.name
    stem = p.stem
    parent = p.parent.name.lower()

    cls = "normal"
    if "benign" in parent or "benign" in fname.lower():
        cls = "benign"
    elif "malignant" in parent or "malignant" in fname.lower():
        cls = "malignant"

    mask_candidate = p.parent / f"{stem}_mask{p.suffix}"
    if not mask_candidate.exists():
        mask_candidate = p.parent / f"{stem}_mask_1{p.suffix}"
    mask_path = str(mask_candidate) if mask_candidate.exists() else None

    meta_list.append({
        "image_id": fname,
        "file_path": str(p),
        "mask_path": mask_path,
        "ground_truth_label": cls,
        "has_mask": mask_path is not None
    })

df_meta = pd.DataFrame(meta_list)
print(f"Total Cohort Ultrasound Scans: {len(df_meta)}")
print("\nSonographic Class Distribution:")
print(df_meta["ground_truth_label"].value_counts())
print(f"Scans with Paired Segmentation Masks: {df_meta['has_mask'].sum()} / {len(df_meta)}")


## 2. Ultrasound Image Quality Assessment (IQA) Formulation
Ultrasound image quality features unique acoustic parameters:
- **Speckle Index ($C_s$):** Ratio of standard deviation to mean in homogeneous tissue ($C_s = \sigma / \mu$).
- **Equivalent Number of Looks (ENL):** Measure of speckle suppression ($\text{ENL} = \mu^2 / \sigma^2$).
- **Contrast-to-Noise Ratio (CNR):** Normalized lesion-to-background contrast using ground truth masks.
- **Laplacian Variance & Tenengrad:** Measures lesion margin definition.


In [ ]:
# Cell 5: Ultrasound IQA & Speckle Metric Functions
def compute_laplacian_variance(gray):
    return float(np.var(cv2.Laplacian(gray, cv2.CV_64F)))

def compute_tenengrad(gray):
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.mean(gx**2 + gy**2))

def compute_rms_contrast(gray):
    norm = gray.astype(np.float64) / 255.0
    return float(np.std(norm))

def compute_michelson_contrast(gray):
    p1 = float(np.percentile(gray, 1))
    p99 = float(np.percentile(gray, 99))
    return float((p99 - p1) / (p99 + p1 + 1e-6))

def compute_shannon_entropy(gray):
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    return float(entropy(hist, base=2))

def compute_snr_mad(gray):
    mean_val = float(np.mean(gray))
    lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    mad = float(np.median(np.abs(lap - np.median(lap))))
    sigma_n = mad / 0.6745
    if sigma_n < 1e-6:
        return 45.0
    snr = mean_val / (sigma_n + 1e-6)
    return float(np.clip(20.0 * np.log10(max(snr, 1e-6)), -10.0, 50.0))

def compute_ultrasound_speckle(gray):
    h, w = gray.shape
    patch = gray[int(h * 0.3):int(h * 0.7), int(w * 0.3):int(w * 0.7)]
    m = float(np.mean(patch))
    s = float(np.std(patch))
    if m < 1e-4:
        return 0.0, 0.0
    cs = s / m
    enl = (m**2) / (s**2 + 1e-6)
    return float(cs), float(enl)

def compute_cnr(gray, mask):
    if mask is None:
        return 0.0
    bin_mask = mask > 0
    lesion = gray[bin_mask]
    bg = gray[~bin_mask]
    if len(lesion) < 10 or len(bg) < 10:
        return 0.0
    m_les, m_bg = np.mean(lesion), np.mean(bg)
    v_les, v_bg = np.var(lesion), np.var(bg)
    return float(abs(m_les - m_bg) / np.sqrt(v_les + v_bg + 1e-6))

def extract_busi_iqa(path, mask_path=None):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return None
    
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) if (mask_path and os.path.exists(mask_path)) else None
    
    lap = compute_laplacian_variance(gray)
    ten = compute_tenengrad(gray)
    rms = compute_rms_contrast(gray)
    mic = compute_michelson_contrast(gray)
    ent = compute_shannon_entropy(gray)
    snr = compute_snr_mad(gray)
    cs, enl = compute_ultrasound_speckle(gray)
    cnr = compute_cnr(gray, mask)

    norm_lap = np.clip(np.log1p(lap) / 8.0, 0.0, 1.0)
    norm_contrast = np.clip(rms * 4.0, 0.0, 1.0)
    norm_snr = np.clip((snr - 5.0) / 25.0, 0.0, 1.0)
    norm_ent = np.clip(ent / 8.0, 0.0, 1.0)
    composite = float(0.30 * norm_lap + 0.25 * norm_contrast + 0.25 * norm_snr + 0.20 * norm_ent)
    
    tier = "High_Quality" if composite >= 0.65 else ("Medium_Quality" if composite >= 0.45 else "Degraded")
    
    return {
        "laplacian_variance": lap,
        "tenengrad_gradient": ten,
        "speckle_index": cs,
        "enl": enl,
        "cnr": cnr,
        "rms_contrast": rms,
        "michelson_contrast": mic,
        "shannon_entropy": ent,
        "snr_db": snr,
        "composite_quality_score": composite,
        "quality_tier": tier
    }


In [ ]:
# Cell 6: Run IQA Extraction across Ultrasound Scans
sample_n = min(len(df_meta), 300)
eval_df = df_meta.head(sample_n).copy()

iqa_rows = []
for idx, row in eval_df.iterrows():
    m = extract_busi_iqa(row["file_path"], row["mask_path"])
    if m:
        m["image_id"] = row["image_id"]
        iqa_rows.append(m)

df_iqa = pd.DataFrame(iqa_rows)
df_full = pd.merge(eval_df, df_iqa, on="image_id")

# Simulate CNN model confidence
np.random.seed(42)
df_full["cnn_confidence"] = np.clip(0.54 + 0.36 * df_full["composite_quality_score"] + np.random.normal(0, 0.04, len(df_full)), 0.40, 0.99)

print("BUSI Ultrasound IQA Feature Extraction Completed.")
df_full[["ground_truth_label", "speckle_index", "enl", "cnr", "laplacian_variance", "composite_quality_score"]].head()


## 3. Statistical Analysis & Visual Inspection
We present descriptive quality metrics, analyze the distribution of acoustic speckle and contrast, and compare high-clarity ultrasound scans with acoustic-dropout degraded scans.


In [ ]:
# Cell 7: Statistical Summary & Diagnostic Visualizations
print("--- BUSI Ultrasound Descriptive Statistics ---")
display(df_full[["laplacian_variance", "speckle_index", "enl", "cnr", "rms_contrast", "shannon_entropy", "snr_db", "composite_quality_score"]].describe())

# 1. Class Distribution
plt.figure(figsize=(7, 4))
sns.countplot(data=df_full, x="ground_truth_label", hue="ground_truth_label", palette="mako", legend=False)
plt.title("BUSI: Ultrasound Cohort Class Distribution", fontsize=12, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "01_busi_class_distribution.png"), dpi=200)
plt.show()

# 2. Metric Distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_full["laplacian_variance"], kde=True, ax=axes[0, 0], color="purple")
axes[0, 0].set_title("Sharpness (Laplacian Variance)", fontweight="bold")

sns.histplot(df_full["speckle_index"], kde=True, ax=axes[0, 1], color="teal")
axes[0, 1].set_title("Acoustic Speckle Index (Cs = Std/Mean)", fontweight="bold")

sns.histplot(df_full["shannon_entropy"], kde=True, ax=axes[1, 0], color="darkblue")
axes[1, 0].set_title("Shannon Entropy (bits)", fontweight="bold")

sns.histplot(df_full["snr_db"], kde=True, ax=axes[1, 1], color="forestgreen")
axes[1, 1].set_title("Signal-to-Noise Ratio (dB)", fontweight="bold")
plt.suptitle("BUSI Ultrasound: Quality Metric Distributions", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "02_busi_metric_distributions.png"), dpi=200)
plt.show()

# 3. Best vs Degraded Scans Visual Montage
sorted_df = df_full.sort_values(by="composite_quality_score", ascending=False)
best_rows = sorted_df.head(3)
worst_rows = sorted_df.tail(3)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, (_, row) in enumerate(best_rows.iterrows()):
    im = cv2.imread(row["file_path"], cv2.IMREAD_GRAYSCALE)
    axes[0, i].imshow(im, cmap="gray")
    axes[0, i].set_title(f"HIGH QUALITY\nScore: {row['composite_quality_score']:.2f} | Speckle: {row['speckle_index']:.2f}", color="green", fontweight="bold")
    axes[0, i].axis("off")

for i, (_, row) in enumerate(worst_rows.iterrows()):
    im = cv2.imread(row["file_path"], cv2.IMREAD_GRAYSCALE)
    axes[1, i].imshow(im, cmap="gray")
    axes[1, i].set_title(f"ACOUSTIC DROPOUT\nScore: {row['composite_quality_score']:.2f} | Speckle: {row['speckle_index']:.2f}", color="red", fontweight="bold")
    axes[1, i].axis("off")

plt.suptitle("Visual Comparison: High-Quality vs Degraded/Dropout Ultrasound Scans", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "03_best_vs_worst_ultrasound.png"), dpi=200)
plt.show()


## 4. Learned Decision Tree: Quality-Aware Adaptive Clinical Routing
We train an explainable Decision Tree classifier that fuses the **IQA feature vector** with **CNN prediction confidence** to make automated, safety-critical clinical routing decisions.


In [ ]:
# Cell 8: Train Learned Decision Tree & Export Routing Metadata
def assign_clinical_routing_ground_truth(row):
    q = row["composite_quality_score"]
    conf = row["cnn_confidence"]
    lap = row["laplacian_variance"]
    contrast = row["rms_contrast"]
    snr = row["snr_db"]
    
    if q < 0.25 or lap < 20.0:
        return "Reject_Reacquire"
    elif (contrast < 0.12 or snr < 11.0) and lap >= 30.0:
        return "Route_Image_Enhancement"
    elif q >= 0.60 and conf >= 0.80:
        return "Accept_Direct_Prediction"
    else:
        return "Expert_Human_Review"

df_full["clinical_decision_target"] = df_full.apply(assign_clinical_routing_ground_truth, axis=1)

feature_cols = ["composite_quality_score", "laplacian_variance", "speckle_index", "enl", "cnr", "rms_contrast", "shannon_entropy", "snr_db", "cnn_confidence"]
X = df_full[feature_cols].values
y = df_full["clinical_decision_target"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y if len(np.unique(y)) > 1 else None)

dt_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=3, criterion="gini", random_state=42)
dt_model.fit(X_train, y_train)

acc = dt_model.score(X_test, y_test)
print(f"Learned Decision Tree Test Accuracy: {acc * 100:.2f}%
")

print("--- Decision Logic Rules ---")
print(export_text(dt_model, feature_names=feature_cols))

imp_df = pd.DataFrame({"Feature": feature_cols, "Importance": dt_model.feature_importances_}).sort_values(by="Importance", ascending=False)
print("--- Feature Importances ---")
display(imp_df)

output_csv = os.path.join(OUTPUT_DIR, "busi_quality_and_routing_metadata.csv")
df_full.to_csv(output_csv, index=False)
print(f"[SUCCESS] Exported enriched clinical metadata for CNN & Tree training to: {output_csv}")
